In [1]:
import sys
import os
import pandas as pd
from threading import Thread
from multiprocessing import Process, Manager
import datetime
import uuid
import json
import _pickle as pickle
import time
import traceback
from psycopg2 import extras
from binance import AsyncClient, BinanceSocketManager # For MarginCallback
import asyncio # For MarginCallback
import numpy as np
import jwt
import websocket
import pandas as pd

upper_dir = os.path.dirname("/home/trade_core/")
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger
from exchange_plugin.binance_plug import InitBinanceAdaptor
from exchange_plugin.upbit_plug import InitUpbitAdaptor
from etc.register_monitor_msg import RegisterMonitorMsg
from etc.db_handler.postgres_client import InitDBClient

In [2]:
logging_dir = "/home/trade_core/"
with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)
proc_n = 1
# node = config['node']
node = 'trade_core1'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_markets = config['node_settings'][node]['enabled_markets']

# db dict
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [4]:
class InitTrigger:
    def __init__(self, admin_id, node, get_premium_df, enabled_markets, register_monitor_msg, redis_client, db_dict, logging_dir):
        self.admin_id = admin_id
        self.node = node
        self.get_premium_df = get_premium_df
        self.enabled_markets = enabled_markets
        self.register_monitor_msg = register_monitor_msg
        self.redis_client = redis_client
        self.db_dict = db_dict
        self.db_client = InitDBClient(**{**self.db_dict, 'database': 'trade_core'})
        self.logging_dir = logging_dir
        self.logger = KimpBotLogger(logging_dir, "trigger")

    def is_table_empty(conn, table_name):
        with conn.cursor() as cur:
            cur.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cur.fetchone()[0]
            return count == 0

    def get_column_names(conn, table_name):
        with conn.cursor() as cur:
            cur.execute("""
                SELECT column_name 
                FROM information_schema.columns 
                WHERE table_schema = 'public' AND table_name = %s
                ORDER BY ordinal_position;
            """, (table_name,))
            return [row[0] for row in cur.fetchall()]

    def load_user_info_config(self, table_name='user_info'):
        # First check whether the table is empty
        
        pass


In [5]:
def is_table_empty(conn, table_name):
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cur.fetchone()[0]
        return count == 0
    
def get_column_names(conn, table_name):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name 
            FROM information_schema.columns 
            WHERE table_schema = 'public' AND table_name = %s
            ORDER BY ordinal_position;
        """, (table_name,))
        return [row[0] for row in cur.fetchall()]

In [6]:
test_client = InitDBClient(**{**db_dict, 'database': 'trade_core'})
test_client.create_all_table()

InitDBClient|SCHEMA: trade_core already exists.
InitDBClient|TABLE: user_info already exists.
InitDBClient|TABLE: exchange_config created.


In [7]:
conn = test_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

In [30]:
column_list = get_column_names(conn, 'user_info')
pd.DataFrame(columns=column_list)

,id,user_uuid,email,telegram_id,telegram_name,registered_datetime,status,alarm_num,alarm_period,remark


In [29]:
test_client.pool.putconn(conn)

In [7]:
conn = test_client.get_conn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

In [11]:
sql = """
SELECT *
FROM user_info
INNER JOIN exchange_config ON user_info.user_uuid = exchange_config.user_uuid;
"""
curr.execute(sql)
result = curr.fetchall()
pd.DataFrame(result)


,id,user_uuid,email,telegram_id,telegram_name,registered_datetime,status,alarm_num,alarm_period,remark,...,target_market_margin_call,origin_market_margin_call,target_market_safe_reverse,origin_market_safe_reverse,target_market_risk_threshold_p,origin_market_risk_threshold_p,repeat_limit_p,repeat_limit_direction,repeat_num_limit,on_off
0,1,test_uuid,ckddjs116@gmail.com,1390695186,None,2023-12-18 15:54:54.893783,None,1,1,None,...,None,2,True,True,None,None,4.0,UPWARD,None,True
1,2,test_uuid,ckddjs116@gmail.com,1390695186,None,2023-12-18 15:54:58.902161,None,1,1,None,...,None,2,True,True,None,None,4.0,UPWARD,None,True


In [11]:
# For Adding user_info

sql = """
INSERT INTO user_info (user_uuid, email, telegram_id, telegram_name, registered_datetime, status, alarm_num, alarm_period, remark) 
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
val = ['test_uuid', 'ckddjs116@gmail.com', 1390695186, None, datetime.datetime.utcnow(), None, 1, 1, None]
curr.execute(sql, val)
conn.commit()

In [9]:
# For adding exchange_config

sql = """
INSERT INTO exchange_config (user_uuid, registered_datetime, service_datetime_end, target_market_code, origin_market_code, target_market_uid, origin_market_uid,
target_market_referral_use, origin_market_referral_use, target_market_cross, target_market_leverage, origin_market_cross, origin_market_leverage, target_market_margin_call, origin_market_margin_call,
target_market_safe_reverse, origin_market_safe_reverse, target_market_risk_threshold_p, origin_market_risk_threshold_p, repeat_limit_p, repeat_limit_direction, repeat_num_limit,
on_off, remark)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

datetime_now = datetime.datetime.utcnow()
service_datetime_end = datetime_now + datetime.timedelta(days=7)
val = ['test_uuid', datetime.datetime.utcnow(), service_datetime_end, 'UPBIT_SPOT/KRW', 'OKX_USD_M/USDT', None, None,
       False, False, None, None, True, 4, None, 2, True, True, None, None, 4, 'UPWARD', None, True, None]
curr.execute(sql, val)
conn.commit()



In [32]:
conn.rollback()